In [ ]:
# Training

_ = Local(
    epochs = 1,
    beta = 0.1,
    optimizer = bnb.optim.Adam8bit(model.parameters(), lr=2e-5, eps=1e-8)
)

_.scheduler = get_linear_schedule_with_warmup(
    _.optimizer,
    num_warmup_steps = 0,
    num_training_steps = len(train_loader) * _.epochs
)

def calculate_DPO_loss(
    model_preferred_logprob,
    model_dispreferred_logprob,
    ref_preferred_logprob,
    ref_dispreferred_logprob,
    beta=0.5
):
    preferred_relative_logprob = model_preferred_logprob - ref_preferred_logprob
    dispreferred_relative_logprob = model_dispreferred_logprob - ref_dispreferred_logprob

    reward_accuracies = (preferred_relative_logprob > dispreferred_relative_logprob).float().mean()
    reward_margins = (preferred_relative_logprob - dispreferred_relative_logprob).mean()

    loss = -F.logsigmoid(beta * (preferred_relative_logprob - dispreferred_relative_logprob)).mean()

    return loss, preferred_relative_logprob.mean(), dispreferred_relative_logprob.mean(), reward_accuracies, reward_margins

def get_log_prob(logits, labels, prompt_lengths):
    log_probs = F.log_softmax(logits, dim=-1).to(device)
    token_log_probs = torch.gather(log_probs, -1, labels.unsqueeze(-1)).squeeze(-1).to(device)

    batch_size, seq_len = labels.shape
    response_mask = torch.arange(seq_len, device=device).unsqueeze(0) >= prompt_lengths.unsqueeze(1)
    response_mask = response_mask.float()

    response_log_probs = (token_log_probs * response_mask).sum(dim=-1)
    response_lengths = response_mask.sum(dim=-1).clamp(min=1)
    return response_log_probs / response_lengths

def format_time(elapsed):
    '''
    Takes a time in seconds and returns a string hh:mm:ss
    '''
    # Round to the nearest second.
    elapsed_rounded = int(round((elapsed)))

    # Format as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))

# Training loop
def train(model, ref_model, tokenizer, optimizer, scheduler, train_loader, epochs=1, beta=0.1):
    model.gradient_checkpointing_enable() # Memory optimization
    model.train()
    ref_model.eval()

    t0 = time.time()

    for epoch in range(epochs):
        for step, batch in enumerate(train_loader):
            batch = {k: v.to(device) for (k, v) in batch.items()}
            optimizer.zero_grad()

            model_preferred_logits = model(
                input_ids=batch['prompt_preferred_ids'],
                attention_mask=batch['prompt_preferred_mask']
            ).logits

            model_preferred_logprob = get_log_prob(
                model_preferred_logits,
                batch['prompt_preferred_ids'],
                batch['prompt_lengths']
            )

            model_dispreferred_logits = model(
                input_ids=batch['prompt_dispreferred_ids'],
                attention_mask=batch['prompt_dispreferred_mask']
            ).logits

            model_dispreferred_logprob = get_log_prob(
                model_dispreferred_logits,
                batch['prompt_dispreferred_ids'],
                batch['prompt_lengths']
            )

            with torch.no_grad():
                ref_preferred_logits = ref_model(
                    input_ids=batch['prompt_preferred_ids'],
                    attention_mask=batch['prompt_preferred_mask']
                ).logits

                ref_preferred_logprob = get_log_prob(
                    ref_preferred_logits,
                    batch['prompt_preferred_ids'],
                    batch['prompt_lengths']
                )

                ref_dispreferred_logits = ref_model(
                    input_ids=batch['prompt_dispreferred_ids'],
                    attention_mask=batch['prompt_dispreferred_mask']
                ).logits

                ref_dispreferred_logprob = get_log_prob(
                    ref_dispreferred_logits,
                    batch['prompt_dispreferred_ids'],
                    batch['prompt_lengths']
                )

            loss, preferred_relative_logprob, dispreferred_relative_logprob, reward_accuracies, reward_margins = calculate_DPO_loss(
                model_preferred_logprob,
                model_dispreferred_logprob,
                ref_preferred_logprob,
                ref_dispreferred_logprob,
                beta=beta
            )

            loss.backward()
            optimizer.step()
            scheduler.step()

            append_remote("losses.txt", str(loss.item()))

            if step % 10 == 0:
                step_data = dict(
                    loss=loss.item(),
                    preferred_relative_logprob=preferred_relative_logprob.item(),
                    dispreferred_relative_logprob=dispreferred_relative_logprob.item(),
                    reward_accuracy=reward_accuracies.item(),
                    reward_margin=reward_margins.item()
                )

                print(f"{format_time(time.time() - t0)} - {epoch}:{step}/{len(train_loader)} - {step_data}")
                save_model(model, tokenizer)
                torch.cuda.empty_cache()

torch.cuda.empty_cache()
train(model, ref_model, tokenizer, _.optimizer, _.scheduler, train_loader, epochs=_.epochs, beta=_.beta)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


0:00:12 - 0:0/2500 - {'loss': 0.6508378982543945, 'preferred_relative_logprob': -4.6447319984436035, 'dispreferred_relative_logprob': -5.582110404968262, 'reward_accuracy': 0.625, 'reward_margin': 0.9373780488967896}
0:02:13 - 0:10/2500 - {'loss': 0.6055099964141846, 'preferred_relative_logprob': -1.0395936965942383, 'dispreferred_relative_logprob': -3.147771120071411, 'reward_accuracy': 0.75, 'reward_margin': 2.108177423477173}
0:04:14 - 0:20/2500 - {'loss': 0.6394644975662231, 'preferred_relative_logprob': 0.02430558204650879, 'dispreferred_relative_logprob': -1.3407899141311646, 'reward_accuracy': 0.625, 'reward_margin': 1.3650954961776733}
0:06:12 - 0:30/2500 - {'loss': 0.6193560361862183, 'preferred_relative_logprob': -2.3587121963500977, 'dispreferred_relative_logprob': -4.453257083892822, 'reward_accuracy': 0.75, 'reward_margin': 2.0945451259613037}
0:08:05 - 0:40/2500 - {'loss': 0.665410041809082, 'preferred_relative_logprob': -2.3509864807128906, 'dispreferred_relative_logprob